#Naive Bayes setup
#Prior
#Likelihood
#Prediction
#Why it is called “naive”
#Smoothing

In [41]:
#IR:  Naive Bayes

In [2]:
# Naive Bayes Review

## Goal
#Classify a document into a class such as spam or ham.

In [5]:
# Step A: Naive Bayes setup
# Purpose: prepare labeled training documents and a test document
# Code meaning: create a small dataset with two classes, spam and ham
# Result: a simple classification example

In [6]:
train_docs = [
    ("spam", "free prize win"),
    ("spam", "claim prize now"),
    ("ham", "project meeting today"),
    ("ham", "team schedule today")
]

test_doc = "free prize today"

print(train_docs)
print(test_doc)

[('spam', 'free prize win'), ('spam', 'claim prize now'), ('ham', 'project meeting today'), ('ham', 'team schedule today')]
free prize today


In [7]:
# Step B: Inspect the training data
# Purpose: separate class labels and text
# Code meaning: loop through each labeled document
# Result: understand the structure of the training set

In [8]:
for label, text in train_docs:
    print("Label:", label, "| Text:", text)

Label: spam | Text: free prize win
Label: spam | Text: claim prize now
Label: ham | Text: project meeting today
Label: ham | Text: team schedule today


In [9]:
# Step C: Count class frequencies
# Purpose: count how many documents belong to each class
# Code meaning: count the labels in the training data
# Result: the number of spam and ham documents

In [10]:
from collections import Counter

class_counts = Counter()

for label, text in train_docs:
    class_counts[label] += 1

print(class_counts)

Counter({'spam': 2, 'ham': 2})


In [13]:
# Step D: Prior probabilities
#Prior = 단어 보기 전 클래스의 기본 확률
# Purpose: compute the probability of each class before looking at the words
# Code meaning: divide each class count by the total number of documents
# Result: prior probabilities for spam and ham

In [14]:
total_docs = len(train_docs)

priors = {}

for label, count in class_counts.items():
    priors[label] = count / total_docs

print(priors)

{'spam': 0.5, 'ham': 0.5}


In [20]:
#Step E: Collect words by class
#Purpose: group words according to their class
#Code meaning: split each training document into words and add them to the correct class list
#Result: a list of words for each class


#class_words stores all words for each class.
#class_word_counts counts how often each word appears in each class.

In [21]:
#class_words 클래스별 단어를 그냥 모아놓은 리스트
class_words = {
    "spam": [],
    "ham": []
}

for label, text in train_docs:
    words = text.split()
    class_words[label].extend(words)

print(class_words)

{'spam': ['free', 'prize', 'win', 'claim', 'prize', 'now'], 'ham': ['project', 'meeting', 'today', 'team', 'schedule', 'today']}


In [22]:
#class_word_counts클래스별 단어를 그냥 모아놓은 리스트
from collections import Counter

class_word_counts = {}

for label, words in class_words.items():
    class_word_counts[label] = Counter(words)

print(class_word_counts)

{'spam': Counter({'prize': 2, 'free': 1, 'win': 1, 'claim': 1, 'now': 1}), 'ham': Counter({'today': 2, 'project': 1, 'meeting': 1, 'team': 1, 'schedule': 1})}


In [23]:
# Step F: Build the vocabulary
# Purpose: collect all unique words from the training data
# Code meaning: combine words from all classes into one vocabulary set
# Result: a set of unique words and its size

In [24]:
vocab = set()

for label, words in class_words.items():
    vocab.update(words)

print(vocab)
print("Vocabulary size:", len(vocab))

{'now', 'today', 'claim', 'free', 'meeting', 'schedule', 'project', 'team', 'win', 'prize'}
Vocabulary size: 10


In [26]:
# Step G: Count total words in each class
# Purpose: find the total number of word occurrences in each class
# Code meaning: count the length of each class word list
# Result: the total number of words for spam and ham

In [27]:
total_words_per_class = {}

for label, words in class_words.items():
    total_words_per_class[label] = len(words)

print(total_words_per_class)

{'spam': 6, 'ham': 6}


In [28]:
# Step H: Likelihood with Laplace smoothing
# Purpose: compute how likely a word is in each class
# Code meaning: use word counts, total class word counts, and vocabulary size
# Result: conditional probabilities such as P(word | spam) and P(word | ham)

In [32]:
vocab_size = len(vocab)

p_prize_given_spam = (class_word_counts["spam"]["prize"] + 1) / (total_words_per_class["spam"] + vocab_size)
p_prize_given_ham = (class_word_counts["ham"]["prize"] + 1) / (total_words_per_class["ham"] + vocab_size)

p_today_given_spam = (class_word_counts["spam"]["today"] + 1) / (total_words_per_class["spam"] + vocab_size)
p_today_given_ham = (class_word_counts["ham"]["today"] + 1) / (total_words_per_class["ham"] + vocab_size)

print("P(prize | spam) =", p_prize_given_spam)
print("P(prize | ham) =", p_prize_given_ham)
print("P(today | spam) =", p_today_given_spam)
print("P(today | ham) =", p_today_given_ham)

P(prize | spam) = 0.1875
P(prize | ham) = 0.0625
P(today | spam) = 0.0625
P(today | ham) = 0.1875


In [33]:
# Step I: Prepare the test document
# Purpose: split the test document into words
# Code meaning: tokenize the test document by spaces
# Result: a list of words from the test document

In [34]:
test_words = test_doc.split()
print(test_words)

['free', 'prize', 'today']


In [35]:
# Step J: Initialize class scores with priors
# Purpose: start each class score with its prior probability
# Code meaning: assign the prior probability to each class
# Result: initial scores for spam and ham

In [36]:
class_scores = {}

for label in priors:
    class_scores[label] = priors[label]

print(class_scores)

{'spam': 0.5, 'ham': 0.5}


In [38]:
#문서 내용은 아직 안 보고, 클래스 자체 빈도만 봤을 때는 둘 다 똑같다
#Before looking at the wrds in the document, the two classes are equally likely based on their class frequencies

In [40]:
test_words = test_doc.split()
print(test_words)

class_scores = {}

for label in priors:
    class_scores[label] = priors[label]

vocab_size = len(vocab)

for label in class_scores:
    for word in test_words:
        likelihood = (class_word_counts[label][word] + 1) / (total_words_per_class[label] + vocab_size)
        class_scores[label] *= likelihood

print(class_scores)

predicted_label = max(class_scores, key=class_scores.get)
print("Predicted class:", predicted_label)

['free', 'prize', 'today']
{'spam': 0.000732421875, 'ham': 0.0003662109375}
Predicted class: spam
